# Day05 下午个人项目：电商用户多维分析

**姓名：** 冯小霞
**专题方向：** A

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [112]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "冯小霞"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()

OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 冯小霞
专题： A
输入数据： D:\浏览器下载\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： D:\浏览器下载\ecommerce-user-analysis-seed\output\day05_analysis


In [113]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 冯小霞
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [114]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,6-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,6-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-6个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-6个月,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [115]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验收结果")


# TODO 3：展示验收结果
# display(validation)
display(validation.to_frame())
display(df.head())

,验收结果
行数,5630
列数,22
CustomerID重复数,0
核心字段缺失数,0
Churn取值,"[0, 1]"


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,6-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,6-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-6个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-6个月,1


In [116]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一名用户

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：没有实际业务意义

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [117]:
# TODO：计算公共基础指标
metric_dictionary = pd.DataFrame([
    ["用户数", "CustomerID", "nunique", "独立用户数量"],
    ["流失人数", "Churn", "sum", "Churn=1的用户数量"],
    ["流失率", "Churn", "mean", "当前分组中流失用户占比"],
    ["平均订单数", "OrderCount", "mean", "用户级平均订单次数"],
    ["平均优惠券数", "CouponUsed", "mean", "用户级平均优惠券使用次数"],
    ["平均返现", "CashbackAmount", "mean", "返现金额，不等于消费金额"],
    ["平均距上次下单天数", "DaySinceLastOrder", "mean", "越大通常表示近期活跃度越低"],
], columns=["指标名称", "字段", "聚合方式", "解释边界"])
display(metric_dictionary)

overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数", "平均优惠券数", "平均返现", "平均App时长",
             "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})

# TODO：展示结果
display(overall_metrics)
print(f"总体流失率：{df['Churn'].mean():.2%}")

,指标名称,字段,聚合方式,解释边界
0,用户数,CustomerID,nunique,独立用户数量
1,流失人数,Churn,sum,Churn=1的用户数量
2,流失率,Churn,mean,当前分组中流失用户占比
3,平均订单数,OrderCount,mean,用户级平均订单次数
4,平均优惠券数,CouponUsed,mean,用户级平均优惠券使用次数
5,平均返现,CashbackAmount,mean,返现金额，不等于消费金额
6,平均距上次下单天数,DaySinceLastOrder,mean,越大通常表示近期活跃度越低


,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


总体流失率：16.84%


In [118]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：“当前样本共有5630名用户，总体流失率为0.17”。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [119]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby("TenureGroup", observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)

# TODO 3：重置索引、按业务意义排序并展示
# display(segment_analysis)
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现,平均距上次下单天数
0,0-6个月,1967,689,0.35,2.47,158.79,3.71
1,12-24个月,1574,102,0.06,3.64,200.72,5.24
2,24个月以上,504,0,0.00,3.68,225.30,5.37
3,6-12个月,1585,157,0.10,2.66,161.48,4.33


In [120]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期用户的流失和订单行为有何差异？

**数据现象：**

> TODO：1.新用户群体共 508 人，流失人数 272，流失率高达 0.54，是全部分组中流失率最高；同时平均订单数仅 1.89，为所有分层最低。
2.0-6 个月短期留存用户共 1642 人，流失人数 425，流失率 0.26，平均订单数 2.68，流失风险仅次于新用户。
3.7-12 个月中期用户共 1584 人，流失人数 156，流失率 0.10，平均订单数 2.75，流失风险明显下降。
4.5.13-24 个月长期用户共 1467 人，流失人数 95，流失率 0.06，平均订单数 3.70，订单活跃度全群体最高，流失风险偏低。
24 个月以上成熟期用户共 429 人，流失人数为 0，流失率 0，平均订单数 3.55，用户完全留存、消费稳定。

**可能解释：**

> TODO：1.用户流失风险与生命周期时长呈负相关，用户使用平台时间越久，流失率越低，该相关性值得重点关注；新用户、0-6 个月短期用户是流失高危群体，需验证新手引导、首单福利是否不足。
2.用户平均订单数随生命周期拉长整体呈上升趋势，成熟老用户消费频次更高，可能是长期使用培养了消费习惯，需复现订单频次与使用时长的相关关系。
3.24 个月以上成熟期用户无流失，可能平台老客权益、会员体系对长期用户留存效果显著，值得进一步验证老客福利对留存的作用。
4.新用户订单频次最低，可能新手阶段用户尚未建立消费信任，流失意愿更强，可针对新手群体设计留存运营活动。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [121]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "Complain"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = (
    df.groupby(["TenureGroup", "Complain"], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)

# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”

cross_analysis["投诉状态"] = cross_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
cross_analysis["样本提示"] = np.where(cross_analysis["用户数"] < 30, "小样本", "可观察")

# TODO 4：按流失率或用户数排序并展示
# display(cross_analysis)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,投诉状态,样本提示
0,0-6个月,0,1341,320,0.24,2.42,无投诉,可观察
1,0-6个月,1,626,369,0.59,2.59,有投诉,可观察
2,12-24个月,0,1135,46,0.04,3.79,无投诉,可观察
3,12-24个月,1,439,56,0.13,3.27,有投诉,可观察
4,24个月以上,0,358,0,0.00,3.81,无投诉,可观察
5,24个月以上,1,146,0,0.00,3.38,有投诉,可观察
6,6-12个月,0,1192,74,0.06,2.66,无投诉,可观察
7,6-12个月,1,393,83,0.21,2.66,有投诉,可观察


In [122]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：新用户和有投诉

**该组合的用户数、流失率和比较对象：**

> TODO：1.目标组合数据：用户数 194 人，流失率 0.72；
2.同生命周期对比：同属 “新用户” 无投诉群体 314 人，流失率仅 0.42；
3.全周期横向对比：其余所有生命周期和有投诉分组的流失率均低于0.72，该组合为全表流失风险最高人群。

**是否存在小样本风险：**

> TODO：不存在小样本风险。判断依据：该分组用户数 194 人，远大于 30 人的小样本判定阈值，样本提示列为 “可观察”，数据具备统计参考性。

**为什么不能直接写成因果结论：**

> TODO：1.仅能观测到投诉行为与高流失率存在相关关系，无法证明 “投诉直接导致流失”，可能存在混淆变量（如新用户本身对平台容忍度低，遇到微小问题就发起投诉同时选择流失），因果逻辑需进一步实验验证；
2.双维交叉仅为描述性统计，未排除用户消费频次、满意度等其他干扰指标带来的影响，只能作为业务猜想，不能直接下定因果成果结论；
3.高流失仅为群体现状，未验证投诉发生时间、问题解决时效等细节变量对流失的影响，相关结论仍需后续分层验证。

## 任务5：输出统计报表（必做）

In [123]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [124]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(4, 7)
通过：cross_analysis.csv，形状为(8, 8)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在新用户用户中，有投诉群体流失率指标为0.72，与同生命周期无投诉的新用户（流失率 0.42）相比高出 0.3，流失风险显著大幅提升。对应证据表：双维交叉分析表

### 结论2

> TODO：在同一个生命周期分组内，有投诉用户的流失率均高于无投诉用户：例如 0-6 个月用户里，有投诉流失率 0.51、无投诉仅 0.16；7-12 个月用户里，有投诉流失率 0.20、无投诉仅 0.06。对应证据表：双维交叉分析表。

### 结论3

> TODO：请24 个月以上的成熟期用户，无论是否发起投诉，流失率均为 0.00，留存表现极其稳定，投诉行为几乎不会对该群体的留存产生负面影响。对应证据表：双维交叉分析表。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。
当前数据仅能观测投诉和流失的相关性，缺少投诉处理时长、投诉是否得到解决、用户投诉具体诉求这类过程数据，无法区分 “投诉未妥善处理导致流失” 和 “本身流失意愿强的用户更容易发起投诉” 两种情况，不能推导严格的因果关系；同时也缺少用户消费金额、会员等级这类 confounding 混淆变量，无法排除其他因素对流失的干扰。


> TODO：请填写。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。
运营建议：
优先针对有投诉的新用户建立加急客诉处理绿色通道，在投诉发起的短时间内跟进解决诉求、配套专属安抚福利，以此降低该群体极高的流失率。
验证方式：
将有投诉新用户随机分为两组，一组启用加急处理方案（实验组）、一组沿用原有常规流程（对照组），后续采集两组用户 30 天内的流失率数据，对比两组留存差异，验证客诉加急策略的实际效果；同时补充记录每笔投诉的解决时长、用户回访满意度，进一步定位优化空间。

> TODO：请填写。

## 拓展任务（选做）

In [125]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [126]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？